# 售后政策应答的多范式实证对比

## 项目简介

同一套售后政策库、同一份系统提示词、同一批评测样本，分别用三种智能体范式处理，
用可复现的数据回答一个问题：**在这个"按规则办事"的客服场景里，检索与自治到底买回了什么，代价是多少？**

三种模式只允许"编排方式"这一个变量变化：

| 模式 | 框架类 | 工具 | 代表的能力档位 |
|---|---|---|---|
| A | `SimpleAgent` | 无 | 直接依赖模型自身知识作答 |
| B | `FunctionCallAgent` | `search_policy` | 原生函数调用检索依据 |
| C | `ReActAgent` | `search_policy`、`list_policies`、`check_compliance` | 多步推理 + 出稿后自检红线 |

## 项目信息

- 来源：Hello-Agents 进阶篇毕业设计
- 日期：2026-09
- 框架：HelloAgents 0.2.9

## 第一部分：环境配置

依赖装在 Python 3.10+ 环境中，模型走 OpenAI 兼容协议，配置写在工程根目录的 `.env`。

In [1]:
# 首次运行前先安装依赖（已装可跳过）
# !pip install -q -r requirements.txt

import sys
from pathlib import Path

# 保证 notebook 在工程根目录打开时能导入 src 包
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # 从当前工作目录逐级向上找 .env，密钥不入库

from src.agents import MODES, AgentRunner
from src.evaluator import load_samples, run_evaluation
from src.retriever import PolicyRetriever
from src.tools import CheckComplianceTool, ListPoliciesTool, SearchPolicyTool, build_registry

print("环境就绪")

环境就绪


## 第二部分：政策检索与工具定义

先验检索，再验智能体。检索是所有模式共用的地基：地基不准，
后面的对比就分不清"是没查到"还是"查到了没说对"。

In [2]:
retriever = PolicyRetriever(PROJECT_ROOT / "data" / "knowledge_base")

print("知识库概况：")
for key, value in retriever.stats().items():
    print(f"  {key}：{value}")

print("\n典型检索（不需要启动智能体即可看到依据）：")
for query in ["订单三天还没发货", "退款什么时候到账", "收到的包裹少了一件"]:
    hits = retriever.search(query, top_k=2)
    summary = "；".join(f"{h.rule_id or h.name}（{h.score}）" for h in hits) or "未命中"
    print(f"  「{query}」 -> {summary}")

知识库概况：
  规则条数：38
  片段总数：58
  业务域：['价格保护 / 发票凭证', '合规限制', '投诉与赔偿', '支付与退款', '物流履约', '特殊品类', '话术与表达', '退换货与三包', '通用规定']
  词项总数：1511

典型检索（不需要启动智能体即可看到依据）：
  「订单三天还没发货」 -> LOG-001（4.9939）；附：物流类规则速查（4.195）
  「退款什么时候到账」 -> PAY-001（7.9843）；LIM-001（6.5121）
  「收到的包裹少了一件」 -> LOG-003（5.8214）；RET-003（5.6286）


In [3]:
# 三个工具都可以单独验证：它们是纯函数，不依赖模型即可产出结果
print("【search_policy】")
print(SearchPolicyTool(retriever).run({"query": "发货超过 48 小时怎么处理"})[:400])

print("\n【list_policies】")
print(ListPoliciesTool(retriever).run({"domain": "退换货"})[:400])

print("\n【check_compliance】")
print(CheckComplianceTool().run({"draft": "请您放心，退款明天一定到账，我们一定给您换新。"}))

【search_policy】
命中 3 条政策依据（按相关度排序）：

── 第 1 条 ──
【LOG-001】发货时效与超时处理（来源：02-物流履约规则.md）
| 规则编号 | LOG-001 |
| 适用场景 | 用户催发货、订单超过承诺时效未发出 |
| 责任部门 | 仓储物流部 |
| 时效标准 | 付款成功后 48 小时内安排发货（特殊商品以商品页说明为准） |
| 关联规则 | GEN-003、PAY-001、LOG-005 |

### 判定标准

| 条件 | 判定 |
|---|---|
| 订单状态为"待发货"或"已付款未出库" | 进入本规则 |
| 距付款成功未超 48 小时 | 属正常范围，告知预计时间 |
| 距付款成功已超 48 小时 | **判定为发货超时**，进入加急流程 |
| 商品页标注了预售、定制等特殊发货期 | 以商品页说明为准，不适用 48 小时标准 |

### 处理

【list_policies】
共 6 条规则（业务域含「退换货」）：
- RET-001 七天无理由退货［退换货与三包］
- RET-002 质量问题退换处理［退换货与三包］
- RET-003 少件漏发处理［退换货与三包］
- RET-004 三包期内维修［退换货与三包］
- RET-005 换货流程与时效［退换货与三包］
- RET-006 不适用无理由退货的品类［退换货与三包］

提示：拿到编号后，可用 search_policy 检索该编号对应的条文原文。

【check_compliance】
发现 1 处红线风险：
- [时效/结果类]（依据 LIM-001 / LIM-002）命中片段：一定到账

请改写上述表述后再定稿：改为说明处理动作与后续跟进方式，不给出具体时间点、金额或审核结论。


## 第三部分：三种模式的智能体构建

In [4]:
for mode, info in MODES.items():
    print(f"模式 {mode}｜{info['label']}｜{info['agent']}")
    print(f"    能力档位：{info['capability']}")

print("\n各模式实际装配的工具：")
for mode, tool_mode in (("A", "none"), ("B", "search"), ("C", "full")):
    _, tools = build_registry(retriever, tool_mode)
    print(f"  模式 {mode}：{[tool.name for tool in tools] or '（无工具）'}")

模式 A｜无检索直答｜SimpleAgent（不带工具）
    能力档位：直接依赖模型自身知识作答
模式 B｜单跳检索｜FunctionCallAgent + search_policy
    能力档位：原生函数调用检索政策依据
模式 C｜多步自治｜ReActAgent + 三个工具
    能力档位：多步推理、先规划再检索、出稿后自检红线

各模式实际装配的工具：
  模式 A：（无工具）
✅ 工具 'search_policy' 已注册。
  模式 B：['search_policy']
✅ 工具 'search_policy' 已注册。
✅ 工具 'list_policies' 已注册。
✅ 工具 'check_compliance' 已注册。
  模式 C：['search_policy', 'list_policies', 'check_compliance']


## 第四部分：功能演示

同一条工单跑三遍。肉眼可见的差异是：没有依据时敢不敢下结论、
有依据时会不会越界承诺、多步自检能不能把越界的表述拦下来。

In [5]:
DEMO_TICKET = "我买的蓝牙耳机订单 20260709001，到现在三天还没发货，客服也没人回。我现在很着急，能不能赶紧发货？"

runner = AgentRunner(retriever, quiet=True)

for mode in ("A", "B", "C"):
    result = runner.run(DEMO_TICKET, mode)
    print("=" * 74)
    print(f"模式 {mode}｜{MODES[mode]['label']}")
    print(f"耗时 {result.elapsed_ms} ms｜模型调用 {result.llm_calls} 次｜工具调用 {result.tool_calls} 次")
    print("-" * 74)
    print(result.answer)
    if result.error:
        print(f"[提示] {result.error}")

模式 A｜无检索直答
耗时 9979 ms｜模型调用 1 次｜工具调用 0 次
--------------------------------------------------------------------------
<tool_calls>
<tool_call>invoke<tool_sep>
<tool_call>name>search_rule<tool_sep>
<tool_call>parameter<key>category, query</parameter><parameter value>LOG 发货时效 三天未发货</parameter>
</tool_call>


模式 B｜单跳检索
耗时 9088 ms｜模型调用 2 次｜工具调用 1 次
--------------------------------------------------------------------------
**客户回复**：

您好，非常抱歉让您久等了，理解您着急的心情。

关于您的订单 20260709001，我查到了您的情况。平台规定付款成功后 48 小时内安排发货，您的订单已超过这个时效，确实属于发货超时，我这边已经帮您登记了加急催发工单，会同步给仓储物流部门优先处理。

由于发货时间会根据仓库出库排期、库存状况等实际情况安排，我无法给您承诺具体的发货时点，但请您放心，我会持续跟进您的订单进度，有更新会第一时间通知您。

同时，关于您提到客服回复不及时的问题，我也已记录并会向相关部门反馈改进。

如果您等待过程中有任何其他问题，欢迎随时联系我。再次为给您带来的不便表示歉意。

---

**内部处理建议**：

| 项目 | 内容 |
|---|---|
| **问题类型** | 物流问题 |
| **依据规则** | LOG-001 |
| **建议动作** | 1. 核实订单实际付款时间与当前状态；2. 查询仓库发货排期及库存情况；3. 发起加急催发工单并记录超时原因；4. 跟进物流部门处理结果后及时同步客户 |
| **需要核实的信息** | 订单付款成功时间、订单当前状态（待发货/已付款未出库等）、商品页是否有特殊发货期标注、仓库库存状态 |
| **风险提醒** | ❌ 不可承诺具体发货时间点；❌ 不可承诺快递当天揽收；❌ 超时赔付金额需人工审批，不得自行承诺 |


模式 C｜多步自治
耗时 19004 ms｜模型调用 2 次｜工具调用 1 次
--------------------------------------------------------------------------
客户回复：您好，非常抱歉让您久等了！已查询到订单 20260709001，按平台规则付款成功后 48 小时内应安排发货（特殊商品以商品页说明为准），目前订单已超过该时效。我已将您的订单标记为加急催发，并同步提交仓库核实当前库存与发货排期，会尽快为您推进。同时，如果您对等待时间难以接受，也可以告诉我，我们将为您转接退款处理流程。后续有任何进度会第一时间通过消息通知您，感谢您的耐心与理解。

依据规则：LOG-001、GEN-003

内部建议：问题类型：物流问题；建议动作：核对订单实际付款时间与当前状态，查询仓库库存及发货排期，升级加急催发流程并记录超时原因，向客户同步进度；需要核实的信息：订单状态、付款时间、库存情况；风险提醒：不可承诺具体发货时间点或快递公司当天揽收，超时赔付需人工审批，不得直接承诺金额。

风险提醒：不承诺具体发货时点，不承诺快递当天揽收，超时赔付需走人工审批流程，仅告知区间性处理意向。


## 第五部分：性能评估

指标全部由字符串规则判定，不让模型当裁判：

| 指标 | 判据 |
|---|---|
| 未完成率 | 达到最大步数仍未定稿 |
| 红线违规率 | 命中数据集 `must_not_contain` 断言 |
| 编号编造率 | 出现知识库编号表之外的规则编号 |
| 依据覆盖率 | `expected_rules` 是否被引用 |
| 追问触发率 | `should_ask` 样本中是否真的追问 |

完整结果见 `outputs/comparison.md`（随项目提交）。
下面先跑前 5 条确认链路，确认无误后把 `samples[:5]` 改成 `samples` 即可跑全量。

In [6]:
samples = load_samples(PROJECT_ROOT / "data" / "eval" / "eval_samples.yaml")
print(f"评测样本共 {len(samples)} 条（正常诉求 / 边界 / 信息缺失 / 未覆盖 / 失败样本固化）")

# 演示只跑前 5 条，刻意输出到独立目录：仓库里的 outputs/comparison.md 是 30 条全量结果，
# 不能被这里的演示覆盖掉
outcome = run_evaluation(
    retriever=retriever,
    samples=samples[:5],
    modes=["A", "B", "C"],
    out_dir=PROJECT_ROOT / "outputs" / "notebook-demo",
)

评测样本共 30 条（正常诉求 / 边界 / 信息缺失 / 未覆盖 / 失败样本固化）


  [A] case_001 ok （11316ms，1 次调用，555 字）


  [A] case_002 ok （6005ms，1 次调用，454 字）


  [A] case_003 ok （9605ms，1 次调用，388 字）


  [A] case_004 ok （13210ms，1 次调用，533 字）


...

  [C] case_001 ok （27505ms，3 次调用，309 字）


  [C] case_002 ok （24167ms，3 次调用，400 字）


  [C] case_003 ok （21536ms，3 次调用，337 字）


  [C] case_004 ok （38893ms，3 次调用，401 字）


  [C] case_005 违规 （22805ms，3 次调用，448 字）


In [7]:
from IPython.display import Markdown, display

# 展示本次演示（前 5 条）的对比结果
display(Markdown((PROJECT_ROOT / "outputs" / "notebook-demo" / "comparison.md").read_text(encoding="utf-8")))

# 仓库中的 outputs/comparison.md 是 30 条全量结果，由命令行 `python -m src.evaluator` 生成
print("全量报告：outputs/comparison.md")

# 三种模式的对比报告

> 本文件由 `src/evaluator.py` 生成，指标全部由字符串规则判定，可逐条复查。

## 一、总体指标

| 指标 | 模式 A（无检索直答） | 模式 B（单跳检索） | 模式 C（多步自治） |
|---|---|---|---|
| 未完成率 | 0.0% | 0.0% | 0.0% |
| 红线违规率 | 0.0% | 20.0% | 20.0% |
| 规则化复核违规率 | 0.0% | 60.0% | 0.0% |
| 编号编造率 | 0.0% | 0.0% | 0.0% |
| 依据覆盖率 | 20.0% | 100.0% | 100.0% |
| 追问触发率 | —（无样本） | —（无样本） | —（无样本） |

## 二、成本指标

| 指标 | 模式 A | 模式 B | 模式 C |
|---|---|---|---|
| 平均 LLM 调用 | 1.00 | 2.00 | 3.00 |
| 平均工具调用 | 0.00 | 1.80 | 2.00 |
| 平均耗时(ms) | 9,595.60 | 18,305.20 | 26,981.20 |
| 平均回复长度 | 482.00 | 693.40 | 379.00 |
| 异常数 | 0 | 0 | 0 |
| 样本数 | 5 | 5 | 5 |

## 三、问题明细（便于复核）

| 样本 | 模式 | 红线违规 | 编造编号 | 复核命中 | 异常 |
|---|---|---|---|---|---|
| case_003 | B | — | — | 时效/结果类（LIM-001 / LIM-002）：承诺补发或赔付 | — |
| case_004 | B | 一定到账 | — | 时效/结果类（LIM-001 / LIM-002）：一定到账 | — |
| case_005 | B | — | — | 金额类（LIM-003）：299 元 | — |
| case_005 | C | 今天就退 | — | — | — |

## 四、判定口径说明

- **未完成率**：达到最大步数仍未定稿（框架返回固定话术）的比例。
  这一项必须与违规率合并解读：没答完的样本不可能违规，直接对比会低估自治模式的风险。
- **红线违规率**：回复命中数据集 `must_not_contain` 任一短语即计违规（数据集自带断言）。
- **规则化复核违规率**：用生成阶段同款红线正则复核一遍，用于交叉验证判据一致性。
- **编号编造率**：回复中出现知识库编号表之外的 `XXX-000` 形式编号即计编造。
- **依据覆盖率**：`expected_rules` 被回复引用的比例，仅在标注了期望规则的样本上统计。
- **追问触发率**：`should_ask` 为真的样本中，回复出现「请/麻烦 + 提供/告知/确认」类请求的比例；
  未标注 `should_ask` 的样本不参与该指标（避免把客服用语误判为追问）；


全量报告：outputs/comparison.md


## 第六部分：总结与展望

### 实现的功能

- 三种可复现的编排方式，共用同一套政策库与提示词，差异只在编排
- 四个脚本可判定的指标 + 规则化红线复核，结论可逐条复查
- 政策检索采用规则级切分 + BM25，工程自包含、可离线运行

### 遇到的挑战

- 框架的 ReAct 用行内正则解析动作，`Finish[...]` 一旦跨行就会解析成空串；
  已在提示词中强制单行，并在外层加入轨迹兜底
- 同一批工具在原生函数调用与文本解析两条路径下的参数名不同（`query` 与 `input`），
  工具必须同时兼容，否则 ReAct 模式永远拿到空参数
- 批量评测会触发供应商限流，已加入指数退避重试，避免脏数据污染结论

### 未来改进方向

- 引入向量检索与重排，观察混合检索能否提升长尾诉求的覆盖
- 把确定性流程作为第四档对照（当前三档都是自治路径）
- 扩充样本到多领域、多模型，检验结论的普适性